In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

AR1_NAME = "AR(1)-GRW-S"
ARW4_NAME = "ARW4"
MODEL_TAG = "stagewise_ar1_grw_arw4"
Y = 20
YEARS = np.arange(Y + 1)

palette = ["#EF476F","#118AB2","#06D6A0","#073B4C","#FFD166"]
EMP_COLOR, ARW4_COLOR, AR1_COLOR = palette[:3]
model_specs = [("Empirical",EMP_COLOR,"-"),(AR1_NAME,AR1_COLOR,"-."),(ARW4_NAME,ARW4_COLOR,"--")]
stage_order = ["years_1_4","years_5_7","years_8_20"]
stage_titles = {"years_1_4": "Years 1-4","years_5_7": "Years 5-7","years_8_20": "Years 8-20"}

OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
sns.set_context("talk",font_scale=.82)
sns.set_style("ticks")
plt.rcParams.update({"figure.facecolor": "white","axes.facecolor": "white","savefig.facecolor": "white","font.family": "sans-serif","font.sans-serif": ["Helvetica","Arial","DejaVu Sans"],"pdf.fonttype": 42,"ps.fonttype": 42,"axes.titlelocation": "left","axes.titlepad": 8,"legend.frameon": False})

In [ ]:
def combine(name):
    ar1 = pd.read_csv(INPUT_DIR / name); arw4 = pd.read_csv(INPUT_DIR / f"arw4_{name}")
    emp = ar1.loc[ar1.source.eq("empirical")].assign(model="Empirical")
    ar1 = ar1.loc[ar1.source.eq("simulated")].assign(model=AR1_NAME)
    arw4 = arw4.loc[arw4.source.eq("simulated")].assign(model=ARW4_NAME)
    return pd.concat([emp,ar1,arw4],ignore_index=True)

def save_figure(fig, stem):
    fig.savefig(OUTPUT_DIR / f"{MODEL_TAG}_{stem}.png",dpi=300,bbox_inches="tight",pad_inches=.1)
    fig.savefig(OUTPUT_DIR / f"{MODEL_TAG}_{stem}.pdf",bbox_inches="tight",pad_inches=.1)
    plt.show(); plt.close(fig)

def panel(ax, label): ax.annotate(label,(-.13,1.03),xycoords="axes fraction",fontsize=16)

def cutoffs(ax, points):
    for point in points: ax.axvline(point,color="gray",linestyle="--",linewidth=1,alpha=.65,zorder=0)

def finish(ax, xlabel, ylabel, title=None, integer_x=True):
    if title: ax.set_title(title)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    if integer_x: ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.grid(True,color=".83",linewidth=1); ax.xaxis.grid(False)
    ax.tick_params(direction="out")
    sns.despine(ax=ax)

def plot_model(ax, d, x, y, model, marker=True, label=True):
    color = dict((m,c) for m,c,_ in model_specs)[model]
    ls = dict((m,s) for m,_,s in model_specs)[model]
    kw = {"color": color,"linestyle": ls}
    if model == "Empirical":
        ax.plot(d[x],d[y],linewidth=10,alpha=.16,zorder=1,solid_capstyle="round",label="_nolegend_",**kw)
        ax.plot(d[x],d[y],linewidth=4,zorder=5,marker="o" if marker else None,markerfacecolor="white",markeredgecolor=color,markeredgewidth=1.5,markersize=6,label=model if label else None,**kw)
    else:
        ax.plot(d[x],d[y],linewidth=2.5,zorder=3,marker="o" if marker else None,markerfacecolor=color,markeredgecolor="white",markeredgewidth=.7,markersize=4.5,label=model if label else None,**kw)

def lines(ax, df, x, y, marker=True):
    for model,_,_ in model_specs:
        d = df.loc[df.model.eq(model)].sort_values(x)
        plot_model(ax,d,x,y,model,marker=marker)

def laplace_pdf(x, mu, alpha): return np.exp(-np.abs(x - mu) / alpha) / (2 * alpha)

def ecdf(x):
    x = np.sort(np.asarray(x)); return x, np.arange(1,len(x) + 1) / len(x)

In [ ]:
canonical = combine("canonical_trajectory_stats.csv")
yearwise = combine("yearwise_productivity_stats.csv")
log_delta = combine("yearwise_log_delta_stats.csv")
laplace = combine("stage_raw_increment_laplace.csv")
laplace_values = combine("stage_raw_increment_values.csv")
qq = combine("lognormal_qq_coordinates.csv")
rank_corr = combine("rank_correlations.csv")
aggregate = combine("aggregate_productivity_values.csv")
wasserstein_ar1 = pd.read_csv(INPUT_DIR / "wasserstein_by_year.csv")
wasserstein_arw4 = pd.read_csv(INPUT_DIR / "arw4_wasserstein_by_year.csv")
ar1_params = pd.read_csv(INPUT_DIR / "stage_grw_params.csv")
arw4_params = pd.read_csv(INPUT_DIR / "arw4_params.csv")

ar1_cutoffs = sorted(ar1_params.end.astype(int).unique())[:-1]
arw4_cutoffs = sorted(arw4_params.end.astype(int).unique())[:-1]
change_points = sorted(set(ar1_cutoffs + arw4_cutoffs))

print(f"Models: Empirical, {AR1_NAME}, {ARW4_NAME}")
print(f"Change points: {change_points}")

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,4),sharex=True)
for i,(a,y,title) in enumerate(zip(ax,["mean","median"],["Mean productivity","Median productivity"])):
    cutoffs(a,change_points); lines(a,canonical,"career_age",y); finish(a,"Year, $t$","Adjusted productivity, $q_t$",title); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"prodSpaceMeanVsMedian")

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,4),sharex=True)
for i,(a,y,title) in enumerate(zip(ax,["mean_log_prod","var_log_prod"],["Mean log productivity","Variance of log productivity"])):
    cutoffs(a,change_points); lines(a,yearwise,"career_age",y); finish(a,"Year, $t$",title,title); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"logCompare")

In [ ]:
quantiles = ["q25_prod","q50_prod","q75_prod","q90_prod","q95_prod"]
fig, ax = plt.subplots(1,5,figsize=(17,3.7),sharex=True)
for i,(a,y,q) in enumerate(zip(ax,quantiles,[25,50,75,90,95])):
    cutoffs(a,change_points); lines(a,yearwise,"career_age",y); finish(a,"Year, $t$","Adjusted productivity",f"Q{q}"); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"quantiles")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(14,4),sharex=True)
for i,(a,y,title) in enumerate(zip(ax,["mean_log_delta","median_log_delta","var_log_delta"],["Mean","Median","Variance"])):
    cutoffs(a,change_points); lines(a,log_delta,"destination_year",y); finish(a,"Destination year, $t$",title,f"{title} log increment"); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"logDeltaMoments")

In [ ]:
ar1_beta = pd.concat([pd.DataFrame({"career_age": np.arange(int(r.start),int(r.end) + 1),"value": r.beta}) for r in ar1_params.itertuples()],ignore_index=True)
arw4_beta = pd.concat([pd.DataFrame({"career_age": np.arange(int(r.start),int(r.end) + 1),"value": r.mode_beta}) for r in arw4_params.itertuples()],ignore_index=True)
fig, ax = plt.subplots(figsize=(7,4))
cutoffs(ax,change_points)
ax.plot(ar1_beta.career_age,ar1_beta.value,color=AR1_COLOR,linestyle="-.",linewidth=2.5,marker="o",markersize=4.5,label=AR1_NAME)
ax.plot(arw4_beta.career_age,arw4_beta.value,color=ARW4_COLOR,linestyle="--",linewidth=2.5,marker="o",markersize=4.5,label=ARW4_NAME)
finish(ax,"Destination year, $t$","State coefficient","State coefficient over time"); panel(ax,"A."); ax.legend(handlelength=3)
fig.tight_layout(); save_figure(fig,"betaOverTime")

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
cutoffs(ax,change_points); lines(ax,rank_corr,"career_age","rank_correlation_with_year0")
finish(ax,"Year, $t$","Spearman correlation","Rank correlation with year 0"); panel(ax,"A."); ax.legend(handlelength=3)
fig.tight_layout(); save_figure(fig,"rankCorr")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(14,4))
for i,(a,stage) in enumerate(zip(ax,stage_order)):
    emp = laplace_values.loc[(laplace_values.model.eq("Empirical")) & (laplace_values.stage.eq(stage)),"value"].to_numpy()
    lim = max(abs(np.quantile(emp,.01)),abs(np.quantile(emp,.99))); bins = np.linspace(-lim,lim,26); grid = np.linspace(-lim,lim,500)
    a.hist(emp,bins=bins,density=True,color=EMP_COLOR,edgecolor=EMP_COLOR,alpha=.22,linewidth=.7,label="Empirical")
    for model,color,ls in model_specs:
        p = laplace.loc[(laplace.model.eq(model)) & (laplace.stage.eq(stage))].iloc[0]
        if model != "Empirical":
            x = laplace_values.loc[(laplace_values.model.eq(model)) & (laplace_values.stage.eq(stage)),"value"].to_numpy()
            a.hist(x,bins=bins,density=True,histtype="step",linewidth=1.3,color=color,linestyle=ls,label=model)
        y = laplace_pdf(grid,p.mu_hat,p.alpha_hat)
        if model == "Empirical": a.plot(grid,y,color=color,linewidth=8,alpha=.15,zorder=1)
        a.plot(grid,y,color="black" if model == "Empirical" else color,linestyle="--" if model == "Empirical" else ls,linewidth=2.2,zorder=4)
    a.axvline(0,color="gray",linestyle=":",linewidth=1); a.set_yscale("log")
    finish(a,"$q_{t+1}-q_t$","Density",stage_titles[stage],integer_x=False); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=2.5)
fig.tight_layout(); save_figure(fig,"stageRawIncrementLaplace")

In [ ]:
aggregations = ["year_20","last_four","full_career"]
titles = ["Year 20","Any last four years","Full career"]
fig, ax = plt.subplots(1,3,figsize=(14,4))
for i,(a,aggregation,title) in enumerate(zip(ax,aggregations,titles)):
    for model,_,_ in model_specs:
        d = qq.loc[(qq.model.eq(model)) & (qq.aggregation.eq(aggregation))].sort_values("theoretical")
        plot_model(a,d,"theoretical","observed",model,marker=False)
    lim = [min(a.get_xlim()[0],a.get_ylim()[0]),max(a.get_xlim()[1],a.get_ylim()[1])]
    a.plot(lim,lim,color="gray",linestyle=":",linewidth=1.2)
    finish(a,"Normal theoretical quantile",r"Standardized $\log_2$ quantile",title,integer_x=False); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"qqlognorm_full")

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(14,4))
for i,(a,aggregation,title) in enumerate(zip(ax,aggregations,titles)):
    for model,_,_ in model_specs:
        x = aggregate.loc[(aggregate.model.eq(model)) & (aggregate.aggregation.eq(aggregation)),"value"]
        xx, yy = ecdf(x); d = pd.DataFrame({"x": xx,"y": yy})
        plot_model(a,d,"x","y",model,marker=False)
    finish(a,"Cumulative adjusted productivity","ECDF",title,integer_x=False); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"threecumdists")

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(11,4),sharex=True)
for i,(a,y,title) in enumerate(zip(ax,["W1_raw","W1_log1p"],["Raw space","Log space"])):
    cutoffs(a,change_points)
    a.plot(wasserstein_ar1.career_age,wasserstein_ar1[y],color=AR1_COLOR,linestyle="-.",linewidth=2.5,marker="o",markersize=4.5,label=AR1_NAME)
    a.plot(wasserstein_arw4.career_age,wasserstein_arw4[y],color=ARW4_COLOR,linestyle="--",linewidth=2.5,marker="o",markersize=4.5,label=ARW4_NAME)
    finish(a,"Year, $t$","Wasserstein distance",title); panel(a,f"{chr(65 + i)}.")
ax[0].legend(loc="best",handlelength=3)
fig.tight_layout(); save_figure(fig,"wassersteinByYear")